# 02 Baseline model

Before building anything fancy, set a benchmark the real model has to beat.

DLS-affected matches are held out entirely here, since they're what gets tested at the end.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/chases.csv")
normal = df[~df["dls"]].copy()

normal["wickets_left"] = 10 - normal["wickets"]
normal["rrr"] = normal["runs_needed"] / np.maximum(normal["balls_left"], 1) * 6
normal["is_ipl"] = (normal["comp"] == "ipl").astype(int)

## Train/test split by match

Every ball in a match shares the same result. Splitting by row would put balls from the same match in both train and test, so the model would effectively have seen the answer. Splitting by `match_id` keeps whole matches on one side. The overlap check must print 0.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(normal, groups=normal["match_id"]))
train, test = normal.iloc[train_idx], normal.iloc[test_idx]

print(train["match_id"].nunique(), "train matches |", test["match_id"].nunique(), "test matches")
print(len(set(train["match_id"]) & set(test["match_id"])), "overlap")

3678 train matches | 920 test matches
0 overlap


## Dumb baseline vs logistic regression

- **Dumb baseline:** always predict the average chase win rate. Any real model must beat this.
- **Logistic regression:** runs needed, balls left, wickets left, required run rate, and whether it's IPL.

Judged mainly on **log loss** and **Brier score**, not accuracy. The project needs well-calibrated probabilities (is "on DLS par" really 50%?), not just the right winner.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score

features = ["runs_needed", "balls_left", "wickets_left", "rrr", "is_ipl"]
X_train, y_train = train[features], train["won"]
X_test, y_test = test[features], test["won"]

p_dumb = np.full(len(y_test), y_train.mean())

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
p_lr = lr.predict_proba(X_test)[:, 1]

for name, p in [("dumb", p_dumb), ("logreg", p_lr)]:
    print(f"{name:7} log loss {log_loss(y_test, p):.4f}  brier {brier_score_loss(y_test, p):.4f}  acc {accuracy_score(y_test, p > 0.5):.3f}")

dumb    log loss 0.6939  brier 0.2504  acc 0.504
logreg  log loss 0.3816  brier 0.1247  acc 0.817
